In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from astropy.visualization import quantity_support
quantity_support()
# from scipy.stats._stats_py import Power_divergenceResult
%matplotlib inline
from spectrum_component_analyser.readers.JWST.instruments import NIRISS, Instrument, NIRSPECLower
from spectrum_component_analyser.readers.JWST.folder_reader import JWSTFolderReader
from spectrum_component_analyser.readers.JWST.target import *
from spectrum_component_analyser.spectrum import spectrum

target : JWSTTarget = GJ486
instrument : Instrument = NIRSPECLower

all_spectra : list[spectrum] = JWSTFolderReader.get_all_spectra(target=target, instrument=instrument)

# print(all_spectra)

flux_matrix = np.array([spec.Fluxes for spec in all_spectra])

# 2. Create a mask: True only if ALL spectra at that wavelength are finite and > 0
# axis=0 looks across the different spectra for each wavelength point
mask = np.all(np.isfinite(flux_matrix) & (flux_matrix > 0) & (np.abs(flux_matrix) < 1e9), axis=0)

example_spectrum = all_spectra[0]

# mask = np.all(np.isfinite(all_spectra.Fluxes)) & np.all(all_spectra.Fluxes > 0)

s : spectrum
for s in all_spectra:
    s[mask].plot(clear=False)
plt.show()

In [ ]:
# draw light curve

indices = []
fluxes = []

s : spectrum
for index, s in enumerate(all_spectra):
    indices.append(index)
    fluxes.append(np.sum(s.Fluxes[mask].value))

fig, ax = plt.subplots(figsize=(8,8))
ax.plot(indices, fluxes)
plt.show()

for s in all_spectra:
    s.normalise_flux()

In [ ]:
"""
find the standard deviation of each point
find an averaged spectrum
analyse that average spectrum using a (now correct) Pearson chi-squared test
"""

min_included_integration_index = 0
max_included_integration_index = 1000

all_fluxes=[spec.Fluxes for spec in all_spectra[min_included_integration_index:max_included_integration_index]]

average_spectrum = spectrum(wavelengths=all_spectra[0].Wavelengths,
                                 fluxes=np.mean(all_fluxes, axis=0) * all_fluxes[0].unit,
                                 normalised_point=None,
                                 temperature = None,
                                 observational_wavelengths=None,
                                 name="averaged " + target.name)

average_spectrum = average_spectrum[mask]

flux_std_deviation = np.std(all_fluxes, axis=0) * all_fluxes[0].unit

flux_std_deviation = flux_std_deviation[mask]

fig, ax = plt.subplots(figsize=(26,10))
average_spectrum.plot()
plt.errorbar(average_spectrum.Wavelengths, average_spectrum.Fluxes, yerr=flux_std_deviation, ecolor='red', capsize=2)
plt.scatter(average_spectrum.Wavelengths, flux_std_deviation, s=1)
plt.show()


In [ ]:
plt.clf()

from constants import *
from astropy import units as u

from phoenix_grid_creator.spectral_grid import spectral_grid

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))
# fits_file_paths =fits_file_paths[:1]
spec_grid : spectral_grid = spectral_grid.from_local_raw(
    fits_file_paths,
    False,
    observational_wavelengths=average_spectrum.Wavelengths) # need to downsample resolution otherwise the spectral grid won't fit in memory...

# units are getting lost somewhere, idk why
spec_grid.Wavelengths = spec_grid.Wavelengths.to(u.um)
spec_grid.Fluxes *= u.Jy

spec_grid.get_spectrum(spec_grid.T_effs[-1], spec_grid.FeHs[0], spec_grid.Log_gs[0]).plot()

In [ ]:
import numpy as np
from spectrum_component_analyser.chi_squared_minimisation import ChiHelper

number_of_parameters : int = 4 # weight, t, f, l

%matplotlib inline
number_of_components : int = 2

chi : ChiHelper = ChiHelper(
    spec_grid=spec_grid,
    number_of_components=number_of_components,
    number_of_parameters=number_of_parameters,
    observed_spectrum=average_spectrum
)

parameter_bounds = [
        (.0, 2.),
        (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
        (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
        # (target.feh.value - 0.01, target.feh.value + 0.01),
        # (0, 0),
        (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
        # (target.log_g.value - 0.01, target.log_g.value + 0.01)
        # (1.9, 2.1),
    ]

r = chi.get_r(parameter_bounds)

print(r)

In [ ]:
%matplotlib inline

print("simulated / true")

chi.plot(r)

In [ ]:
# unconstrained optimisation
# from spectrum_component_analyser.mcmc.helper import MCMCHelper

# mcmc : MCMCHelper = MCMCHelper(
#     parameter_bounds=parameter_bounds,
#     number_of_components=number_of_components,
#     number_of_parameters=number_of_parameters,
#     observed_spectrum=average_spectrum,
#     spec_grid=spec_grid,
#     n_steps = 2000,
#     n_walkers=64
# )

# sampler, samples = mcmc.run(r)


In [ ]:
# optimisation is constrained so that feh and log g are the same in all components

from spectrum_component_analyser.mcmc.constrained_helper import ConstrainedMCMCHelper

mcmc = ConstrainedMCMCHelper(
    parameter_bounds=parameter_bounds,  # Still [(w_min, w_max), (t_min, t_max), (f_min, f_max), (l_min, l_max)]
    number_of_components=number_of_components,
    observed_spectrum=average_spectrum,
    spec_grid=spec_grid,
    n_steps=10000,
    n_walkers=64
)

# r.x can still be the "flat" 4N array from your global optimizer.
# The class will automatically compress it to the shared FeH/logg format.
sampler, samples = mcmc.run(r)

In [ ]:
%matplotlib inline

mcmc.plot_corner(sampler, samples, true_components=None)

In [ ]:
mcmc.print_parameters(samples=samples)

In [ ]:
%matplotlib widget
mcmc.plot_spectrum(samples)

In [ ]:
import corner
import emcee
from spectrum_component_analyser.spectral_component import spectral_component

plt.rcParams.update({
    'font.size': 24,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
     })
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",
})

def plot_corner(mcmc : ConstrainedMCMCHelper, sampler : emcee.EnsembleSampler, samples : np.ndarray):
    labels = []
    for i in range(mcmc.number_of_components):
        labels += [f"$f_{i+1}$", f"$T_{i+1}$ [K]"]
    labels += ["[Fe/H] [dex]", r"$\log g$ [dex]"]

    main_fill_color = "#FF8B00"
    quantile_line_color = "#AC8DD9"

    # fig, axes = plt.subplots(samples.shape[1], samples.shape[1], figsize=(16, 16))
    
    fontsize = 14
    plt.rc('xtick', labelsize=fontsize - 2)
    plt.rc('ytick', labelsize=fontsize - 2)

    # true_params = []
    # for c in true_components:
    #     true_params += [c.Weight, c.T_eff.value]
    # true_params += [true_feh.value, true_logg.value]
    # print(true_feh)
    # print(true_logg)
    
    # true_params = np.array(true_params, dtype=float)

    truth_color = "black"

    fig = corner.corner(
        samples,
        # truths=true_params,
        truth_color=truth_color,
        # fig=fig,
        labels=labels, 
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        # --- Neatness Settings ---
        color=main_fill_color,
        plot_datapoints=False,         # Removes the scatter points
        plot_density=False,            # Removes the low-res "square" histogram
        fill_contours=True,         # Keeps it looking minimal
        levels=(1 - np.exp(-0.5 * np.arange(1, 4)**2)), # Standard 1, 2, 3 sigma
        contour_kwargs={"colors": [main_fill_color], "linewidths": 1.0},
        hist_kwargs={
            "color": "black",             # This sets the 'edgecolor' when histtype='step'
            "fill": True,                 # Enables the fill
            "facecolor": main_fill_color, # The color inside the histograms
            "edgecolor": "black",         # The outline color
            "linewidth": 1.5,
            "alpha": 0.5                  # Slight transparency often looks cleaner
        },
        title_kwargs={"fontsize": fontsize},
        label_kwargs={"fontsize": fontsize},     
        )
    
    axes = np.array(fig.axes).reshape((samples.shape[1], samples.shape[1]))

    for i in range(samples.shape[1]):
        ax = axes[i, i] # Target the 1D histograms on the diagonal

        current_title = ax.get_title()
        if "T_" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"$T_\text{eff}$ =" + f"{value} K", fontsize=fontsize)
        elif "Fe/H" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"[Fe/H] =" + f"{value} dex", fontsize=fontsize)
        elif "log" in labels[i]:
            value = current_title.split("=")[1]
            ax.set_title(r"$\log g$ =" + f"{value} dex", fontsize=fontsize)
        
        # Corner usually plots quantile lines as 'Line2D' objects
        idx = 0
        for line in ax.get_lines():
            if line.get_color() == truth_color:
                continue
            if (idx == 1):
                line.set_linestyle("-")
            # only count the lines that aren't truth-lines
            idx += 1

            # line.set_color("#705CC8B0")
            # line.set_linestyle("--") # Optional: make them dashed
            # line.set_linewidth(2)     # Optional: make them thicker
    
    for ax in fig.axes:
        for line in ax.get_lines():
            if line.get_color() == truth_color:
                line.set_linewidth(1)
                line.set_markersize(5)
                line.set_zorder(10)

    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'
    # Optional: makes ticks appear on all sides
    plt.rcParams['xtick.top'] = True
    plt.rcParams['ytick.right'] = True

    # fig.suptitle("Posterior distributions for a simulated M dwarf", fontsize=20)
    fig.subplots_adjust(top=0.93)
    plt.savefig("GJ486_NIRSPECLower_2_component.svg", format="svg", bbox_inches='tight')
    plt.show()

    # caterpillar
    # fig, axes = plt.subplots(mcmc.ndim, figsize=(10, 7), sharex=True)
    # chain = sampler.get_chain()
    # for i in range(mcmc.ndim):
    #     ax = axes[i]
    #     ax.plot(chain[:, :, i], "k", alpha=0.3)
    #     ax.set_xlim(0, len(chain))
    #     ax.set_ylabel(labels[i], fontsize=20)
    #     ax.yaxis.set_label_coords(-0.1, 0.5)

    # axes[-1].set_xlabel("Step number")
    # plt.tight_layout()
    # plt.show()

plot_corner(mcmc, sampler, samples)